# Validate and launch the native AHR experiment

Purpose: verify all current-source recovery receipts, run a four-game production pilot and two fresh GNU wiring smoke games, freeze the experiment, and optionally launch the detached controller. Inputs: notebooks 00–04 and passing model/trainer/arena recovery receipts. Outputs: production pilot checkpoints, GNU wiring smoke report, `delivery_validation.json`, immutable configuration, optional tmux launch receipt.

Expected runtime: approximately 5–20 minutes. Checkpoints: `checkpoints/ahr-native-v4-launch-pilot/` and `checkpoints/ahr-native-v4-launch-smoke/`. The pilot uses the real learning/search/GNU path with a reduced update count; its results establish runnable wiring and recovery, not playing strength. Long training requires a separate passing exact-resume test first.

Exact resume: fresh `ahr-go` kernel and Run All with unchanged source/config; the pilot and arena detect their latest valid checkpoints. To launch the long authorized experiment, start the kernel with `AHRV4_LAUNCH=1`. Attach: `tmux attach -t ahr-go-native-v4`; detach: **Ctrl-b, then d**. Default Run All validates and freezes without starting a second writer. Main training and matches continue independently of the user connection after tmux launch.


In [ ]:
from pathlib import Path
import nbformat, os
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT: raise RuntimeError('Run inside AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_native_v4/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]): exec(compile(cell.source,'shared:ahr-export','exec'),globals())
v4_load_upstream(PROJECT,globals())
for name in ('01_native_ahr_and_tactics.ipynb','02_competitive_training.ipynb','03_gnugo_evaluation.ipynb'):
    v4_import_exports(PROJECT/V4_ROOT/name,globals())

v4_import_exports(PROJECT/V4_ROOT/'04_pipeline.ipynb',globals())


## Check current-source recovery evidence

In [ ]:
sources=v4_source_identity(PROJECT)
training_recovery=v4_read_json(v4_report_path(PROJECT,'training_recovery.json'))
arena_recovery=v4_read_json(v4_report_path(PROJECT,'arena_validation.json'))
training_extensions=v4_read_json(v4_report_path(PROJECT,'training_recovery_extensions.json'))
for name,report in [('training',training_recovery),('arena',arena_recovery),('historical_and_stage',training_extensions)]:
    if report.get('status')!='PASS': raise RuntimeError(name+' recovery has not passed')
    identity=report.get('sources',report.get('source_identity'))
    if identity!=sources: raise RuntimeError(name+' recovery source is stale; rerun exact-resume validation')
model_validation=v4_validate_model(PROJECT,device='cuda:1')
pipeline_recovery=v4_validate_pipeline_recovery(PROJECT)
print({'model':model_validation['status'],'training_recovery':training_recovery['status'],
       'arena_recovery':arena_recovery['status'],'pipeline_recovery':pipeline_recovery['status'],'source_notebooks':len(sources)})


## Exercise real learning, search and GNU opposition

In [ ]:
pilot_config=V4TrainConfig(scheduled_games=4,warmup_updates=32,updates_per_game=4,
    device='cuda:1',reanalysis_per_game=2,max_plies=256)
pilot=v4_run_training(PROJECT,'ahr-native-v4-launch-pilot-v3',pilot_config,run_until_games=4)
if pilot.get('games_finished')!=4: raise RuntimeError('Production pilot incomplete')
# Two real games validate wiring in the reserved recovery split; development strength uses32/64.
smoke_run='ahr-native-v4-launch-smoke-v3'
smoke_model,smoke_candidate=v4_eval_pin_model(PROJECT,'ahr-native-v4-launch-pilot-v3',smoke_run,None,'cuda:1')
smoke_model.eval();smoke_model.requires_grad_(False)
smoke_search=V4SearchConfig(time_budget_s=1.)
with torch.inference_mode():
    for _ in range(2):v4_choose_move(smoke_model,initial_state(9,7.5),smoke_search)
torch.cuda.synchronize(next(smoke_model.parameters()).device)
smoke_protocol=V4GNUProtocol(split='recovery_fixture',games=2,seed=910000013)
smoke_players={
    'candidate':FrozenArenaPlayer('candidate',lambda state,budget:v4_choose_move(smoke_model,state,smoke_search),
        {'search':asdict(smoke_search),'ahr_budget_s':1.,'architecture':'native Queen/Worker/Scout','jepa':False},
        {'kind':'ahr_native_v4_checkpoint',**smoke_candidate},model=smoke_model),
    'reference':arena_gnugo_player(PROJECT,'reference',level=1,seed=smoke_protocol.seed,command_timeout_s=3.)}
smoke_identity=v4_eval_identity(PROJECT,smoke_protocol,smoke_players,smoke_candidate)
smoke_snapshot,smoke_stats=v4_eval_run(PROJECT,smoke_run,smoke_identity,smoke_players)
smoke={'status':'COMPLETE' if smoke_snapshot['complete'] else 'IN_PROGRESS','split':'recovery_fixture',
    'eval_run':smoke_run,'games_recorded':len(smoke_snapshot['results']),**smoke_stats,
    'purpose':'Two real production wiring games; no strength claim; excluded from development and final results.'}
v4_atomic_json(v4_report_path(PROJECT,'production_gnu_smoke.json'),{'receipt':smoke,'results':smoke_snapshot['results']})
if smoke.get('games_recorded')!=2: raise RuntimeError('GNU production smoke incomplete')
if smoke.get('unknown')!=0: raise RuntimeError('Production smoke contains UNKNOWN; diagnose before long launch')
print({'pilot_games':pilot['games_finished'],'pilot_optimizer_steps':pilot['optimizer_steps'],
       'gnu_smoke_wins':smoke['wins'],'gnu_smoke_losses':smoke['losses'],'gnu_smoke_unknown':smoke['unknown']})


## Freeze the validated experiment and launch when requested

In [ ]:
backup_wait=time.monotonic()
while True:
    backup_path=v4_report_path(PROJECT,'backup_recovery.json')
    backup_recovery=v4_read_json(backup_path) if backup_path.exists() else {}
    if backup_recovery.get('status')=='PASS' and backup_recovery.get('sources')==sources:break
    if time.monotonic()-backup_wait>1200:raise RuntimeError('Current-source full backup recovery has not passed; long training remains gated')
    time.sleep(2)
validation={'schema':V4_SCHEMA,'status':'PASS','sources':sources,'validated_at_utc':v4_now(),
    'model':model_validation,'training_recovery':training_recovery,'arena_recovery':arena_recovery,
    'pipeline_recovery':pipeline_recovery,'backup_recovery':backup_recovery,'training_extensions':training_extensions,'production_pilot':pilot,'production_gnu_smoke':smoke,
    'scope':'Implementation, native recurrence, exact-resume and real production wiring; no playing-strength claim.'}
if (PROJECT/V4_PROTOCOL_PATH).exists():
    protocol=v4_pipeline_read_protocol(PROJECT)
    if file_sha256(v4_report_path(PROJECT,'delivery_validation.json'))!=protocol['validation_sha256']:
        raise RuntimeError('Frozen validation receipt changed; inspect instead of overwriting provenance')
    v4_atomic_json(v4_report_path(PROJECT,'delivery_recheck.json'),validation)
else:
    v4_atomic_json(v4_report_path(PROJECT,'delivery_validation.json'),validation)
    protocol=v4_pipeline_prepare(PROJECT)
result=v4_launch_detached(PROJECT) if os.environ.get('AHRV4_LAUNCH')=='1' else {
    'status':'VALIDATED_AND_FROZEN','configuration_sha256':protocol['configuration_sha256'],
    'launch_instruction':'Set AHRV4_LAUNCH=1 in a fresh kernel and Run All; inspect any existing tmux session first.'}
print(json.dumps(result,indent=2))
